# 02 · HyP3 y descargas
# *HyP3 and downloads*
**GUAIRA-SAR** · SRL *2026 Venezuela, Colombia and Panama Earthquakes*

🇪🇸 Envía los pares Sentinel-1 a HyP3 (40 m, vectores de mirada) y descarga NISAR, Umbra y Sentinel-1 a Drive.

🇬🇧 Submits Sentinel-1 pairs to HyP3 (40 m, look vectors) and downloads NISAR, Umbra and Sentinel-1 to Drive.

🇪🇸 Corra las celdas **en orden**. Si Colab se reinicia, empiece de nuevo por la celda **0**: los resultados ya guardados en Drive no se pierden.  
🇬🇧 Run the cells **in order**. If Colab restarts, start again from cell **0**: results already saved in Drive are kept.

📝 🇪🇸 Cada celda guarda lo que imprime en `05_documentacion/logs/` y cada figura/tabla se guarda en **español (`_es`) e inglés (`_en`)**.  
📝 🇬🇧 Every cell saves its printed output to `05_documentacion/logs/` and every figure/table is saved in **Spanish (`_es`) and English (`_en`)**.

## 0 · Preparar entorno / *Setup*
🇪🇸 Instala librerías, descarga la última versión del código desde GitHub, monta Drive y registra la versión usada.  
🇬🇧 Installs libraries, pulls the latest code from GitHub, mounts Drive and logs the code version used.

In [ ]:
NOTEBOOK = "02_Descargas_HyP3"
!pip install -q asf_search hyp3_sdk rasterio geopandas shapely contextily pyarrow duckdb boto3 pystac-client requests h5py pyproj scipy huggingface_hub
import os, subprocess, datetime
REPO = "/content/SAR_multifrequency"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/jf-floresriera/SAR_multifrequency", REPO], check=True)
SRC = f"{REPO}/src"
VERSION = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

from google.colab import drive
drive.mount("/content/drive")
RAIZ = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/GUAIRA-SAR"
TRABAJO = "/content/drive/My Drive/InSAR_LaGuaira_Resultados/inventario_2026"
for d in (TRABAJO, f"{RAIZ}/05_documentacion"):
    os.makedirs(d, exist_ok=True)
os.chdir(TRABAJO)
with open(f"{RAIZ}/05_documentacion/versiones_codigo.csv", "a") as f:   # trazabilidad / traceability
    f.write(f"{datetime.datetime.now():%Y-%m-%d %H:%M},{NOTEBOOK},{VERSION}\n")
import sys, importlib
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import comun; importlib.reload(comun)
LOGS = f"{RAIZ}/05_documentacion/logs"
def correr(script):
    """Ejecuta src/<script> y guarda todo lo impreso en 05_documentacion/logs / runs and logs everything."""
    return comun.correr(script, LOGS, SRC, NOTEBOOK, VERSION)
print(f"Código / code: {VERSION} · Carpeta / folder: {os.getcwd()} · Registros / logs: {LOGS}")


## 1 · Credenciales NASA Earthdata / *NASA Earthdata credentials*

In [ ]:
import asf_search as asf, getpass
usuario = input("Usuario Earthdata / username: ")
clave = getpass.getpass("Contraseña / password: ")
session = asf.ASFSession().auth_with_creds(usuario, clave)
print("Autenticación correcta / authenticated")


## 1b · ¿Ya terminó HyP3? / *Has HyP3 finished?*
🇪🇸 Muestra el estado de todos sus trabajos y descarga los terminados (sin repetir). HyP3 borra los productos ~14 días después: descárguelos a tiempo.  
🇬🇧 Shows the status of all your jobs and downloads the finished ones (no duplicates). HyP3 deletes products ~14 days later: download them in time.

In [ ]:
DESCARGAR = True
correr("hyp3_estado.py")

## 2 · Enviar 033A y 025D a HyP3 / *Submit 033A and 025D to HyP3*
🇪🇸 Gasta créditos HyP3; no repite trabajos ya enviados.  
🇬🇧 Uses HyP3 credits; does not resubmit existing jobs.

In [ ]:
ENVIAR_HYP3 = True
correr("s1_pares_hyp3.py")

## 3 · Descargar todo a Drive / *Download everything to Drive*
🇪🇸 `ESPERAR_HYP3 = True` espera a que HyP3 termine (puede tardar horas). Con `False` descarga lo listo y sigue.  
🇬🇧 `ESPERAR_HYP3 = True` waits for HyP3 (may take hours). With `False` it downloads what is ready and continues.

In [ ]:
ESPERAR_HYP3 = False
correr("descargas.py")